In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from movebench.study import Study
from movebench.ssf import grouped_nll, iSSFDataset, get_issf_channel_info, MOVEMENT_FEATURES

In [3]:
study = Study.load('tsavo_lions') # ('5a280544-06db-4c85-9713-842547972edd')
study.download_rasters(splits=["train", "test"],
                        # skip_existing=False
                        )

print(study)

[download_study_rasters] Study: tsavo_lions
[download_study_rasters] Total points: 2154
[download_study_rasters] Splits: ['train', 'test'], Chip groups: ['era5', 'modis', 'high_res'], Tile layers: ['dem', 'dist_water']

[download_study_rasters] Downloading train chips (1330 points)...
[download_chips] Buffer: 19.939 km, split: train, groups: ['era5', 'modis', 'high_res']
[download_chips] Starting download of 1330 chips with 16 workers


[download_chips] Complete: 0 downloaded, 3990 existing, 0 failed

[download_study_rasters] Downloading test chips (824 points)...
[download_chips] Buffer: 19.939 km, split: test, groups: ['era5', 'modis', 'high_res']
[download_chips] Starting download of 824 chips with 16 workers


[download_chips] Complete: 0 downloaded, 2472 existing, 0 failed

[download_study_rasters] Downloading dem tiles...
[download_tile_layer:dem] All 2 tiles already exist, skipping

[download_study_rasters] Downloading dist_water tiles...
[download_tile_layer:dist_water] All 2 tiles already exist, skipping

[download_study_rasters] All downloads complete for tsavo_lions
Study('tsavo_lions')


In [4]:
# Create iSSF dataset
issf_ds = study.issf_dataset(split='train',
                             include_time=True)
print(f"iSSF dataset: {len(issf_ds)} samples")
print(f"Implementation: {type(issf_ds).__module__}.{type(issf_ds).__name__}")

# Load one sample
step_pixels, valid_pixels, transform, crs = issf_ds[0]
print(f"Step pixels shape: {step_pixels.shape}")

# Show channel layout
info = get_issf_channel_info()
print(f"\nChannel layout:")
print(f"  Env features:   channels {info['env'][0]}-{info['env'][1]-1} ({info['n_env']} channels)")
print(f"  sl_:            channel {info['sl_'][0]}")
print(f"  log_sl_:        channel {info['log_sl_'][0]}")
print(f"  cos_ta_:        channel {info['cos_ta_'][0]}")
print(f"  Interactions:   channels {info['interactions'][0]}-{info['interactions'][1]-1} ({info['n_interactions']} channels)")
print(f"  Total:          {info['total']} channels")

[movebench] Computing normalization for 22 layers: ['modis_lst', 'modis_vi', 'modis_ndsi', 'modis_bai', 'era5_land', 'era5', 'era5_land_daily', 'era5_daily', 'dem', 'slope', 'aspect', 'rugosity', 'dist_water', 'orographic_uplift', 'thermal_uplift', 'modis_cvf_tree', 'modis_cvf_nontree', 'modis_cvf_nonveg', 'hii', 'bioclim', 'dynamic_world', 'landsat_ndwi']


Computing stats: 100%|██████████| 1000/1000 [09:30<00:00,  1.75it/s]
/data/vision/beery/scratch/justin/movebench/movebench/utils.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False).astype(bool)


Building feature cache (1294 samples)...
  Processing sample 0/1294
  Processing sample 50/1294
  Processing sample 100/1294
  Processing sample 150/1294
  Processing sample 200/1294
  Processing sample 250/1294
  Processing sample 300/1294
  Processing sample 350/1294
  Processing sample 400/1294
  Processing sample 450/1294
  Processing sample 500/1294
  Processing sample 550/1294
  Processing sample 600/1294
  Processing sample 650/1294
  Processing sample 700/1294
  Processing sample 750/1294
  Processing sample 800/1294
  Processing sample 850/1294
  Processing sample 900/1294
  Processing sample 950/1294
  Processing sample 1000/1294
  Processing sample 1050/1294
  Processing sample 1100/1294
  Processing sample 1150/1294
  Processing sample 1200/1294
  Processing sample 1250/1294
Saved feature cache to /data/vision/beery/scratch/justin/movebench/studies/tsavo_lions/cache/issfdataset_train_eec8c3f9130fa3bc.pt
iSSF dataset: 1294 samples
Implementation: movebench.ssf.iSSFDataset
St

In [5]:
# Inspect movement features for the true endpoint (last column)
print("Movement features for true endpoint (index 100):")
print(f"  sl_     = {step_pixels[info['sl_'][0], -1].item():.2f} m")
print(f"  log_sl_ = {step_pixels[info['log_sl_'][0], -1].item():.4f}")
print(f"  cos_ta_ = {step_pixels[info['cos_ta_'][0], -1].item():.4f}")

print(f"\nMovement features for first pseudo-candidate (index 0):")
print(f"  sl_     = {step_pixels[info['sl_'][0], 0].item():.2f} m")
print(f"  log_sl_ = {step_pixels[info['log_sl_'][0], 0].item():.4f}")
print(f"  cos_ta_ = {step_pixels[info['cos_ta_'][0], 0].item():.4f}")

Movement features for true endpoint (index 100):
  sl_     = -0.50 m
  log_sl_ = 0.2176
  cos_ta_ = 1.0226

Movement features for first pseudo-candidate (index 0):
  sl_     = -0.13 m
  log_sl_ = 0.5653
  cos_ta_ = -0.5147


In [6]:
# Check validity
move_start, move_end = info['movement']
print(f"Movement validity (should be all True): {valid_pixels[move_start:move_end].all().item()}")
print(f"Overall valid ratio: {valid_pixels.float().mean().item():.2%}")

Movement validity (should be all True): True
Overall valid ratio: 91.94%


---
## Train iSSF Model

Train a conditional logistic regression model using the iSSFDataset.
The model learns coefficients for environmental features, movement features,
and their interactions.

In [8]:
# Extract features from iSSFDataset

def extract_ssf_features(dataset, num_workers=8):
    """Extract all features from an SSFDataset into tensors."""
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=num_workers)
    
    all_xs = []
    all_masks = []
    
    for step_pixels, valid_pixels, _, _ in tqdm(loader, desc="Extracting features"):
        all_xs.append(step_pixels)
        all_masks.append(valid_pixels)
    
    # Stack: (N, C, num_candidates+1)
    X = torch.cat(all_xs, dim=0)
    masks = torch.cat(all_masks, dim=0)
    
    # Reshape to (N * (num_candidates+1), C) for regression
    N, C, K = X.shape
    X = X.permute(0, 2, 1).reshape(-1, C)
    masks = masks.permute(0, 2, 1).reshape(-1, C)
    
    # Labels: last candidate (index K-1) is the true endpoint
    y = torch.zeros(N * K, dtype=torch.long)
    y[K-1::K] = 1  # Every K-th element starting from K-1
    
    # Group indices for conditional logistic regression
    starts = torch.arange(0, N * K, K)
    ends = torch.arange(K, N * K + 1, K)
    
    return X, masks, y, starts, ends

# Extract train features
train_issf = study.issf_dataset(split='train',
                                include_time=True)

X_tr, masks_tr, y_tr, starts_tr, ends_tr = extract_ssf_features(train_issf)

print(f"Train features: {X_tr.shape}")
print(f"Train groups: {len(starts_tr)}")

/data/vision/beery/scratch/justin/movebench/movebench/utils.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False).astype(bool)


Loading cached features from /data/vision/beery/scratch/justin/movebench/studies/tsavo_lions/cache/issfdataset_train_eec8c3f9130fa3bc.pt


Extracting features: 100%|██████████| 1294/1294 [00:01<00:00, 1095.77it/s]


Train features: torch.Size([130694, 843])
Train groups: 1294


In [9]:
# Clean features: fill invalid values with 0 (approx mean after normalization)
def clean_features(X, masks):
    X_clean = torch.where(masks, X, torch.tensor(0.0))
    X_clean = torch.nan_to_num(X_clean, nan=0.0, posinf=0.0, neginf=0.0)
    return X_clean

X_tr_clean = clean_features(X_tr, masks_tr)

print(f"Features per sample: {X_tr_clean.shape[1]}")

Features per sample: 843


In [15]:
# Train conditional logistic regression
num_features = X_tr_clean.shape[1]
beta = torch.zeros(num_features, dtype=torch.float32, requires_grad=True)
opt = torch.optim.Adam([beta], 
                       lr=1e-2, 
                    #    weight_decay=1e-3
                       )

for step in range(5000):
    opt.zero_grad()
    loss = grouped_nll(beta, X_tr_clean, y_tr, starts_tr, ends_tr)
    loss.backward()
    opt.step()
    
    if step % 100 == 0:
        with torch.no_grad():
            nll_tr = grouped_nll(beta, X_tr_clean, y_tr, starts_tr, ends_tr).item()
            print(f"Step {step:4d} | Train NLL: {nll_tr:.4f}")

        # if initialized ..
        with torch.no_grad():
            nll_te = grouped_nll(beta, X_te_clean, y_te, starts_te, ends_te).item()
            print(f"Test NLL: {nll_te:.4f}")

Step    0 | Train NLL: 4.5114
Test NLL: 4.5887
Step  100 | Train NLL: 4.1546
Test NLL: 4.8731
Step  200 | Train NLL: 4.1260
Test NLL: 5.0098
Step  300 | Train NLL: 4.1118
Test NLL: 5.1173


KeyboardInterrupt: 

In [11]:
# Inspect learned coefficients
from movebench.config import get_channel_indices, LAYERS

# Build channel names matching the iSSF layout
indices = get_channel_indices()
env_channel_names = []
for name, (start, end) in indices.items():
    layer = LAYERS[name]
    if layer.categorical:
        for i in range(layer.num_classes):
            env_channel_names.append(f"{name}_class{i}")
    else:
        for band in layer.bands:
            env_channel_names.append(f"{name}_{band}")

# Movement feature names
move_names = list(MOVEMENT_FEATURES)

# Interaction names
interact_names = []
for mf in MOVEMENT_FEATURES:
    for env_name in env_channel_names:
        interact_names.append(f"{mf}:{env_name}")

channel_names = env_channel_names + move_names + interact_names

beta_np = beta.detach().numpy()
sorted_idx = abs(beta_np).argsort()[::-1]

print("Top 15 coefficients by magnitude:\n")
for i in sorted_idx[:15]:
    print(f"{channel_names[i]:45s} {beta_np[i]:+.4f}")

print("\n--- Movement feature coefficients ---")
info = get_issf_channel_info()
move_start, move_end = info['movement']
for i in range(move_start, move_end):
    print(f"{channel_names[i]:45s} {beta_np[i]:+.4f}")

Top 15 coefficients by magnitude:



IndexError: list index out of range

## Test Set Evaluation

In [12]:
import time

# Time test set extraction
test_issf = study.issf_dataset(split='test',
                               include_time=True)
print(f"Test dataset: {len(test_issf)} samples")

start = time.time()
X_te, masks_te, y_te, starts_te, ends_te = extract_ssf_features(test_issf)
elapsed = time.time() - start

print(f"Test extraction time: {elapsed:.1f}s")
print(f"Test features: {X_te.shape}")

/data/vision/beery/scratch/justin/movebench/movebench/utils.py:181: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_step_next = df['valid_step'].shift(-1).fillna(False).astype(bool)


Building feature cache (815 samples)...
  Processing sample 0/815
  Processing sample 50/815
  Processing sample 100/815
  Processing sample 150/815
  Processing sample 200/815
  Processing sample 250/815
  Processing sample 300/815
  Processing sample 350/815
  Processing sample 400/815
  Processing sample 450/815
  Processing sample 500/815
  Processing sample 550/815
  Processing sample 600/815
  Processing sample 650/815
  Processing sample 700/815
  Processing sample 750/815
  Processing sample 800/815
Saved feature cache to /data/vision/beery/scratch/justin/movebench/studies/tsavo_lions/cache/issfdataset_test_f0a4550322b40990.pt
Test dataset: 815 samples


Extracting features: 100%|██████████| 815/815 [00:00<00:00, 1076.06it/s]


Test extraction time: 1.0s
Test features: torch.Size([82315, 843])


In [13]:
# Evaluate on test set
X_te_clean = clean_features(X_te, masks_te)

with torch.no_grad():
    nll_te = grouped_nll(beta, X_te_clean, y_te, starts_te, ends_te).item()
    print(f"Test NLL: {nll_te:.4f}")

Test NLL: 5.5685
